In [2]:
from calibration import (
    load_entries,
    load_template_features_fixed_length,
    dtw_cross_costs,
    compute_pr_curve_from_costs,
    find_threshold_best_f1_from_pr,
    plot_pr_curve_from_costs
)
from feature import(
    make_stft_extractor,
    load_features_from_json
)
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

In [3]:
#editable variables
feat_method = "stft"
n_calib = 10
n_temps = 5

In [4]:
#Path to get calib_templates
DATA_26 = Path(r"C:\Users\HP\Desktop\Skripsie data\26")
RESULTS_26 = DATA_26 / "template_selections"

#Path to get templates
DATA_29 = Path(r"C:\Users\HP\Desktop\Skripsie data\29")
RESULTS_29 = DATA_29 / "template_selections"

#File names
bg_file = "Bground_CALIB_26_20230426_075505_background.json"

st_calib_file = "ST_CALIB_26_20230426_075505_single_tone_templates.json"
mt_calib_file = "MT_CALIB_26_20230426_075505_multi_tone_templates.json"
bt_calib_file = "BT_CALIB_26_20230426_075505_burst_tonal_templates.json"

st_temp_file = "20230429_Templates_20230429_102841_single_tone_templates.json"
mt_temp_file = "20230429_Templates_20230429_102841_multi_tone_templates.json"
bt_temp_file = "20230429_Templates_20230429_102841_burst_tonal_templates.json"


In [5]:
#feature frame lengths to test
feat_frame_lengths = [0.048, 0.064, 0.096, 0.128]
frame_len_results = {}

#window frame lengths to test
window_lengths_to_test = [0.1 ,0.2 ,0.3, 0.4, 0.5, 0.6, 0.7] 

call_types = ["st", "mt", "bt"]

#load start and end times for calibration templates
bg_temps = load_entries(RESULTS_26 / bg_file, n_calib)
st_calib_temps = load_entries(RESULTS_26 / st_calib_file, n_calib)
bt_calib_temps = load_entries(RESULTS_26 / mt_calib_file, n_calib)
mt_calib_temps = load_entries(RESULTS_26 / bt_calib_file, n_calib)

In [6]:
threshold_results = {}

for feat_frame_length in feat_frame_lengths:
    #Type of feature extraction method
    if feat_method == "stft":
        #setup the estractor to be used
        extractor = make_stft_extractor(frame_dur = feat_frame_length)  
    window_results = {}

    for window_len in window_lengths_to_test:
        if feat_frame_length < window_len:
            #determine features
            bg_feats = [load_template_features_fixed_length(template_number, extractor, window_len) for template_number in bg_temps]

            st_calib_feats = [load_template_features_fixed_length(template_number, extractor, window_len) for template_number in st_calib_temps]
            mt_calib_feats = [load_template_features_fixed_length(template_number, extractor, window_len) for template_number in mt_calib_temps]
            bt_calib_feats = [load_template_features_fixed_length(template_number, extractor, window_len) for template_number in bt_calib_temps]

            st_temp_feats = load_features_from_json(RESULTS_29 / st_temp_file, extractor, n_temps)
            mt_temp_feats = load_features_from_json(RESULTS_29 / mt_temp_file, extractor, n_temps)
            bt_temp_feats = load_features_from_json(RESULTS_29 / bt_temp_file, extractor, n_temps)

            feats = {
                            "st": (st_calib_feats, st_temp_feats),
                            "mt": (mt_calib_feats, mt_temp_feats),
                            "bt": (bt_calib_feats, bt_temp_feats)
                        }

            window_results[window_len] = {}
            threshold_results[(feat_frame_length, window_len)] = {}
            
            for call_type, (calib_feats, temp_feats) in feats.items():
                #Calculate DTW costs
                call_costs = dtw_cross_costs(temp_feats, calib_feats, extractor.metric)
                bg_costs = dtw_cross_costs(temp_feats, bg_feats, extractor.metric)

                #calculate Precision, Recall for different thresholds
                precision, recall, pr_thresholds = compute_pr_curve_from_costs(call_costs, bg_costs)
                # Find the threshold based on F1 score
                threshold, f1 = find_threshold_best_f1_from_pr(precision, recall, pr_thresholds)

                window_results[window_len][call_type] = f1 #{len: {'call': f1}}
                threshold_results[(feat_frame_length, window_len)][call_type] = threshold

        else:
            #if the stft frame length is longer than the tested window length
            print(f"{feat_frame_length} larger than {window_len}")
    
    #Find best window length base on F1
    best_window_len = max(window_results, key=lambda w: min(window_results[w].values())) #finds the window length of the max F1 score of the minimum F1 scores for different lengths
    best_score = min(window_results[best_window_len].values())
    frame_len_results[feat_frame_length] = (best_window_len, best_score)

#Find the best feature frame length based on F1 scores
best_frame_len = max(frame_len_results, key=lambda f: (frame_len_results[f][1], f))
best_window_len, best_score = frame_len_results[best_frame_len]
best_thresholds = threshold_results[(best_frame_len, best_window_len)]
st_threshold = best_thresholds["st"]
mt_threshold = best_thresholds["mt"]
bt_threshold = best_thresholds["bt"]

print(f"Chosen Frame Length: {best_frame_len}")
print(f"Chosen Window Length: {best_window_len}")
print(f"Best F1 Score: {best_score}")
print(f"st_threshold: {st_threshold}")
print(f"mt_threshold: {mt_threshold}")
print(f"bt_threshold: {bt_threshold}")

0.128 larger than 0.1
Chosen Frame Length: 0.064
Chosen Window Length: 0.5
Best F1 Score: 0.8505747126436781
st_threshold: 0.10806259710540696
mt_threshold: 0.12636192757238074
bt_threshold: 0.1160123240989284


In [7]:
if n_temps == 5:
    vote_fracs = [0.2, 0.4, 0.6, 0.8, 1.0]
if n_temps == 10:
    vote_fracs = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
vote_frac_results = {}

In [ ]:
if feat_method == "stft":
    extractor = make_stft_extractor(frame_dur = best_frame_len)

bg_feats = [load_template_features_fixed_length(template_number, extractor, best_window_len) for template_number in bg_temps]

st_calib_feats = [load_template_features_fixed_length(template_number, extractor, best_window_len) for template_number in st_calib_temps]
mt_calib_feats = [load_template_features_fixed_length(template_number, extractor, best_window_len) for template_number in mt_calib_temps]
bt_calib_feats = [load_template_features_fixed_length(template_number, extractor, best_window_len) for template_number in bt_calib_temps]

st_temp_feats = load_features_from_json(RESULTS_29 / st_temp_file, extractor, n_temps)
mt_temp_feats = load_features_from_json(RESULTS_29 / mt_temp_file, extractor, n_temps)
bt_temp_feats = load_features_from_json(RESULTS_29 / bt_temp_file, extractor, n_temps)

calib_feats = {
    "st": st_calib_feats,
    "mt": mt_calib_feats,
    "bt": bt_calib_feats,
    "bg": bg_feats
}

temp_feats = {
    "st": st_temp_feats,
    "mt": mt_temp_feats,
    "bt": bt_temp_feats
}

call_thresholds = {
    "st": st_threshold,
    "mt": mt_threshold,
    "bt": bt_threshold
}

triggered = {}

for vote_frac in vote_fracs:

    for call_type_calib, calib_feat in calib_feats.items():

        for call_type_temp, temp_feat in temp_feats.items():
            flat_costs = dtw_cross_costs(temp_feat, calib_feat, extractor.metric) #list of costs
            call_costs = np.asarray(flat_costs).reshape(len(temp_feat), len(calib_feat)) #(n_temps, n_calib)
            
            for call_type_thresholds, call_threshold in call_thresholds.items():
                call_votes = (call_costs < call_threshold).sum(axis=0)
                call_triggered = call_votes >= np.ceil(vote_frac * n_temps)
                triggered[(vote_frac, call_type_calib, call_type_temp, call_type_thresholds)] = call_triggered 

call_sets = ["st", "mt", "bt"]

f1_scores = {}   # (vote_frac, temp_type, thr_type) -> f1
for vote_frac in vote_fracs:
    for temp_type in temp_feats:
        for thr_type in call_thresholds:
            pos_trig = np.concatenate([triggered[(vote_frac, call_label, temp_type, thr_type)] for call_label in call_sets])          # every call example
            neg_trig = triggered[(vote_frac, "bg", temp_type, thr_type)]  # background only

            tp = pos_trig.sum()
            fn = (~pos_trig).sum()
            fp = neg_trig.sum()

            p = tp / (tp + fp) if (tp + fp) else 0.0
            r = tp / (tp + fn) if (tp + fn) else 0.0
            f1 = 2 * p * r / (p + r) if (p + r) else 0.0
            f1_scores[(vote_frac, temp_type, thr_type)] = f1
            print(f"vote_frac={vote_frac:.1f} temps={temp_type} thr={thr_type}: P={p:.2f} R={r:.2f} F1={f1:.2f}")

# choose vote_frac: best worst-case F1 across all template/threshold combinations
best_vote_frac = max(vote_fracs, key=lambda v: (min(f1_scores[(v, t, th)] for t in temp_feats for th in call_thresholds), v))
print("Best vote_frac:", best_vote_frac)

vote_frac=0.2 temps=st thr=st: P=1.00 R=0.93 F1=0.97
vote_frac=0.2 temps=st thr=mt: P=0.97 R=0.97 F1=0.97
vote_frac=0.2 temps=st thr=bt: P=1.00 R=0.97 F1=0.98
vote_frac=0.2 temps=mt thr=st: P=1.00 R=0.97 F1=0.98
vote_frac=0.2 temps=mt thr=mt: P=1.00 R=1.00 F1=1.00
vote_frac=0.2 temps=mt thr=bt: P=1.00 R=1.00 F1=1.00
vote_frac=0.2 temps=bt thr=st: P=1.00 R=1.00 F1=1.00
vote_frac=0.2 temps=bt thr=mt: P=0.97 R=1.00 F1=0.98
vote_frac=0.2 temps=bt thr=bt: P=1.00 R=1.00 F1=1.00
vote_frac=0.4 temps=st thr=st: P=1.00 R=0.90 F1=0.95
vote_frac=0.4 temps=st thr=mt: P=1.00 R=0.97 F1=0.98
vote_frac=0.4 temps=st thr=bt: P=1.00 R=0.93 F1=0.97
vote_frac=0.4 temps=mt thr=st: P=1.00 R=0.80 F1=0.89
vote_frac=0.4 temps=mt thr=mt: P=1.00 R=1.00 F1=1.00
vote_frac=0.4 temps=mt thr=bt: P=1.00 R=0.87 F1=0.93
vote_frac=0.4 temps=bt thr=st: P=1.00 R=0.97 F1=0.98
vote_frac=0.4 temps=bt thr=mt: P=1.00 R=1.00 F1=1.00
vote_frac=0.4 temps=bt thr=bt: P=1.00 R=0.97 F1=0.98
vote_frac=0.6 temps=st thr=st: P=1.00 R=0.87 F